# Exploring Exported Knowledge Graph Data

This notebook explores the exported TSV files (nodes.tsv and edges.tsv) from the temporal knowledge graph.

## Setup

First, let's import the necessary libraries and set up the data paths.


In [ ]:
import json
from pathlib import Path

import polars as pl

# Set up paths - adjust these to point to your exported TSV files
# By default, assumes files are in the parent directory
base_dir = Path("/mnt/diskGum/manas/GOMgraph-main/agro_temporal_kg/exported_data")
nodes_file = base_dir / "nodes.tsv"
edges_file = base_dir / "edges.tsv"

print(f"Nodes file: {nodes_file}")
print(f"Edges file: {edges_file}")
print(f"Nodes file exists: {nodes_file.exists()}")
print(f"Edges file exists: {edges_file.exists()}")


: 

## Load Data

Load the TSV files into Polars DataFrames.


In [2]:
# Load nodes
nodes_df = pl.read_csv(
    nodes_file,
    separator="\t",
    null_values=[""],
    try_parse_dates=True,
)

# Load edges
edges_df = pl.read_csv(
    edges_file,
    separator="\t",
    null_values=[""],
    try_parse_dates=True,
)

print(f"Nodes shape: {nodes_df.shape}")
print(f"Edges shape: {edges_df.shape}")
print("\nNodes columns:", nodes_df.columns)
print("\nEdges columns:", edges_df.columns)


Nodes shape: (40, 13)
Edges shape: (77, 13)

Nodes columns: ['uuid', 'type', 'name', 'group_id', 'labels', 'created_at', 'summary', 'attributes', 'source', 'source_description', 'content', 'valid_at', 'entity_edges']

Edges columns: ['uuid', 'type', 'source_node_uuid', 'target_node_uuid', 'group_id', 'created_at', 'name', 'fact', 'episodes', 'expired_at', 'valid_at', 'invalid_at', 'attributes']


## Basic Statistics

Let's get an overview of the data.


In [3]:
print("=== NODES ===")
print(f"Total nodes: {len(nodes_df)}")
print("\nNode types:")
print(nodes_df.group_by("type").agg(pl.count()).sort("count", descending=True))

print("\n=== EDGES ===")
print(f"Total edges: {len(edges_df)}")
print("\nEdge types:")
print(edges_df.group_by("type").agg(pl.count()).sort("count", descending=True))


=== NODES ===
Total nodes: 40

Node types:
shape: (2, 2)
┌──────────┬───────┐
│ type     ┆ count │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ Entity   ┆ 36    │
│ Episodic ┆ 4     │
└──────────┴───────┘

=== EDGES ===
Total edges: 77

Edge types:
shape: (2, 2)
┌────────────┬───────┐
│ type       ┆ count │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ MENTIONS   ┆ 48    │
│ RELATES_TO ┆ 29    │
└────────────┴───────┘


/tmp/ipykernel_721992/2713233052.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  print(nodes_df.group_by("type").agg(pl.count()).sort("count", descending=True))
/tmp/ipykernel_721992/2713233052.py:9: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  print(edges_df.group_by("type").agg(pl.count()).sort("count", descending=True))


In [11]:
nodes_df["summary"][0]

'Melons require consistent watering due to heat from the manure bed beneath them, as heat increases humidity inside the growing enclosure. This humidity must be managed to avoid harm to the plants, which thrive in properly maintained conditions.'

In [12]:
edges_df.head(5)


uuid,type,source_node_uuid,target_node_uuid,group_id,created_at,name,fact,episodes,expired_at,valid_at,invalid_at,attributes
str,str,str,str,str,"datetime[μs, UTC]",str,str,str,str,"datetime[μs, UTC]",str,str
"""cdfaec80-7ed1-4f5c-ab8c-8ced07…","""RELATES_TO""","""6eb4d93c-ce39-41b0-9ed3-7ace47…","""3db26263-2dbe-4ac3-95ce-05e515…","""agro-test-2""",2025-12-23 08:54:49.810531 UTC,"""AFFECTED_BY""","""Melons are affected by the hea…","""[""b6687b77-7871-4a9c-a751-6a81…",null,1880-01-01 00:00:00 UTC,null,"""{""uuid"": ""cdfaec80-7ed1-4f5c-a…"
"""f8ed3949-bca1-4a22-a32b-a884c3…","""RELATES_TO""","""6eb4d93c-ce39-41b0-9ed3-7ace47…","""8c450884-431d-49d2-a66a-8dd6ff…","""agro-test-2""",2025-12-23 08:54:49.810614 UTC,"""AFFECTED_BY""","""Melons can be affected by exce…","""[""b6687b77-7871-4a9c-a751-6a81…",null,1880-01-01 00:00:00 UTC,null,"""{""uuid"": ""f8ed3949-bca1-4a22-a…"
"""d3cc6232-6e2e-4b5d-8ff5-98473b…","""RELATES_TO""","""4d6b83b8-c836-41dd-8efb-0dcb95…","""3db26263-2dbe-4ac3-95ce-05e515…","""agro-test-2""",2025-12-23 08:54:49.810581 UTC,"""CAUSE_OF""","""The manure bed causes heat.""","""[""b6687b77-7871-4a9c-a751-6a81…",null,1880-01-01 00:00:00 UTC,null,"""{""uuid"": ""d3cc6232-6e2e-4b5d-8…"
"""593e348f-c0cc-4336-bd45-492030…","""RELATES_TO""","""3db26263-2dbe-4ac3-95ce-05e515…","""b14208ac-a406-4d2e-b72c-9a0fc2…","""agro-test-2""",2025-12-23 08:54:49.810600 UTC,"""CAUSE_OF""","""Heat causes moisture to form i…","""[""b6687b77-7871-4a9c-a751-6a81…",null,1880-01-01 00:00:00 UTC,null,"""{""uuid"": ""593e348f-c0cc-4336-b…"
"""a3a2f971-0d56-4a16-bedc-1dc997…","""RELATES_TO""","""8c450884-431d-49d2-a66a-8dd6ff…","""8c450884-431d-49d2-a66a-8dd6ff…","""agro-test-2""",2025-12-23 08:55:49.256849 UTC,"""CAUSE_OF""","""Excessive moisture causes high…","""[""4fc8bd27-2e5c-4f00-8273-7b61…",null,1880-01-01 00:00:00 UTC,null,"""{""uuid"": ""a3a2f971-0d56-4a16-b…"


## Explore Entity Nodes

Let's dive deeper into Entity nodes.


In [13]:
entity_nodes = nodes_df.filter(pl.col("type") == "Entity")
print(f"Total Entity nodes: {len(entity_nodes)}")
print("\nEntity nodes sample:")
print(entity_nodes.select(["uuid", "name", "group_id", "summary"]).head(10))


Total Entity nodes: 36

Entity nodes sample:
shape: (10, 4)
┌─────────────────────────────────┬──────────────┬─────────────┬─────────────────────────────────┐
│ uuid                            ┆ name         ┆ group_id    ┆ summary                         │
│ ---                             ┆ ---          ┆ ---         ┆ ---                             │
│ str                             ┆ str          ┆ str         ┆ str                             │
╞═════════════════════════════════╪══════════════╪═════════════╪═════════════════════════════════╡
│ 6eb4d93c-ce39-41b0-9ed3-7ace47… ┆ melons       ┆ agro-test-2 ┆ Melons require consistent wate… │
│ 4d6b83b8-c836-41dd-8efb-0dcb95… ┆ manure bed   ┆ agro-test-2 ┆ Manure beds generate heat requ… │
│ 3db26263-2dbe-4ac3-95ce-05e515… ┆ heat         ┆ agro-test-2 ┆ The text discusses the extreme… │
│ b14208ac-a406-4d2e-b72c-9a0fc2… ┆ moisture     ┆ agro-test-2 ┆ Moisture levels from manure be… │
│ 8c450884-431d-49d2-a66a-8dd6ff… ┆ humidity     

## Explore Episodic Nodes

Let's look at Episodic nodes (episodes/chunks from the source documents).


In [14]:
episodic_nodes = nodes_df.filter(pl.col("type") == "Episodic")
print(f"Total Episodic nodes: {len(episodic_nodes)}")
if len(episodic_nodes) > 0:
    print("\nEpisodic nodes sample:")
    print(
        episodic_nodes.select(
            ["uuid", "name", "group_id", "source", "source_description", "valid_at"]
        ).head(5)
    )


Total Episodic nodes: 4

Episodic nodes sample:
shape: (4, 6)
┌───────────────────────┬─────────────┬─────────────┬────────┬──────────────────────┬──────────────┐
│ uuid                  ┆ name        ┆ group_id    ┆ source ┆ source_description   ┆ valid_at     │
│ ---                   ┆ ---         ┆ ---         ┆ ---    ┆ ---                  ┆ ---          │
│ str                   ┆ str         ┆ str         ┆ str    ┆ str                  ┆ datetime[μs] │
╞═══════════════════════╪═════════════╪═════════════╪════════╪══════════════════════╪══════════════╡
│ b6687b77-7871-4a9c-a7 ┆ test-part_1 ┆ agro-test-2 ┆ text   ┆ These passages       ┆ 1880-01-01   │
│ 51-6a81c9…            ┆             ┆             ┆        ┆ describe tradit…     ┆ 00:00:00     │
│ 9787f6de-5d06-4ac8-92 ┆ test-part_2 ┆ agro-test-2 ┆ text   ┆ These passages       ┆ 1880-01-01   │
│ 69-a41aae…            ┆             ┆             ┆        ┆ describe tradit…     ┆ 00:00:00     │
│ 4fc8bd27-2e5c-4f00-82 ┆ tes

In [15]:
relates_to_edges = edges_df.filter(pl.col("type") == "RELATES_TO")
print(f"Total RELATES_TO edges: {len(relates_to_edges)}")

if len(relates_to_edges) > 0:
    print("\nEdge relationship types:")
    print(relates_to_edges.group_by("name").agg(pl.count()).sort("count", descending=True))
    
    print("\nSample RELATES_TO edges:")
    print(relates_to_edges.select(["name", "fact", "source_node_uuid", "target_node_uuid"]).head(10))


Total RELATES_TO edges: 29

Edge relationship types:
shape: (9, 2)
┌───────────────┬───────┐
│ name          ┆ count │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ AFFECTED_BY   ┆ 10    │
│ CAUSE_OF      ┆ 7     │
│ REQUIRES      ┆ 3     │
│ LOCATED_IN    ┆ 2     │
│ PROTECTS_FROM ┆ 2     │
│ PREFERS       ┆ 2     │
│ OCCURS_AT     ┆ 1     │
│ AffectedBy    ┆ 1     │
│ CONTAINS      ┆ 1     │
└───────────────┴───────┘

Sample RELATES_TO edges:
shape: (10, 4)
┌─────────────┬────────────────────────────┬───────────────────────────┬───────────────────────────┐
│ name        ┆ fact                       ┆ source_node_uuid          ┆ target_node_uuid          │
│ ---         ┆ ---                        ┆ ---                       ┆ ---                       │
│ str         ┆ str                        ┆ str                       ┆ str                       │
╞═════════════╪════════════════════════════╪═══════════════════════════╪══════════════════════════

/tmp/ipykernel_721992/1770010217.py:6: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  print(relates_to_edges.group_by("name").agg(pl.count()).sort("count", descending=True))


## Join Nodes and Edges

Let's create a more readable view by joining edges with node names.


In [ ]:
# Create a mapping of uuid to name for quick lookups
node_names = nodes_df.select(["uuid", "name"]).rename({"name": "node_name"})

# Join edges with source and target node names
edges_with_names = (
    edges_df.join(node_names, left_on="source_node_uuid", right_on="uuid", how="left")
    .rename({"node_name": "source_name"})
    .drop("uuid")
    .join(node_names, left_on="target_node_uuid", right_on="uuid", how="left")
    .rename({"node_name": "target_name"})
    .drop("uuid")
)

print("Edges with node names:")
print(
    edges_with_names.select(
        ["type", "name", "source_name", "target_name", "fact"]
    ).head(10)
)


## Temporal Analysis

Explore temporal aspects of the data (valid_at, created_at).


In [ ]:
# Parse dates if they're strings
nodes_with_dates = nodes_df.with_columns(
    pl.col("created_at").str.to_datetime().alias("created_at_parsed"),
    pl.col("valid_at").str.to_datetime().alias("valid_at_parsed"),
)

print("Nodes by creation date:")
print(
    nodes_with_dates.group_by(
        pl.col("created_at_parsed").dt.date().alias("date")
    )
    .agg(pl.count())
    .sort("date")
)

# Check for temporal edges
if "valid_at" in edges_df.columns:
    edges_with_dates = edges_df.with_columns(
        pl.col("valid_at").str.to_datetime().alias("valid_at_parsed")
    )
    print("\nEdges with valid_at dates:")
    print(
        edges_with_dates.filter(pl.col("valid_at_parsed").is_not_null())
        .select(["name", "fact", "valid_at_parsed"])
        .head(10)
    )
